# 12 - Advanced bridge: change data feed and SCD reasoning

Prerequisites / execution context: Run 00. Dedicated table. Uses the explicitly enabled legacy Delta CDF path for a controlled lab; current docs also describe automatic CDF with separate requirements.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R68: Change data feed](https://docs.databricks.com/aws/en/tables/features/change-data-feed); [R69: AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc); [R26: Delta MERGE](https://docs.databricks.com/aws/en/delta/merge); [R31: Watermarks](https://docs.databricks.com/aws/en/structured-streaming/watermarks).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Enable CDF before writes
CDF exposes changes from the supported retained history, not a permanent archive. Record the start version for your consumer.


In [ ]:
T=f"{SILVER}.cdf_lab"
spark.sql(f"CREATE OR REPLACE TABLE {T} (id INT,name STRING) USING DELTA TBLPROPERTIES (delta.enableChangeDataFeed=true)")
start=spark.sql(f"DESCRIBE HISTORY {T} LIMIT 1").first().version
spark.sql(f"INSERT INTO {T} VALUES (1,'old'),(2,'keep')")
spark.sql(f"UPDATE {T} SET name='new' WHERE id=1")
spark.sql(f"DELETE FROM {T} WHERE id=2")
changes=spark.sql(f"SELECT * FROM table_changes('{T}',{start})")
changes.orderBy("_commit_version","id","_change_type").show(truncate=False)
types={r[0] for r in changes.select("_change_type").distinct().collect()}
assert {"insert","update_preimage","update_postimage","delete"} <= types


## 2. SCD Type 1 versus Type 2
Type 1 keeps current attributes. Type 2 retains effective periods for attribute versions. For a customer moving IT -> FR, sketch one current Type 1 row and two Type 2 rows with non-overlapping validity intervals. Use a source sequence or event timestamp plus a deterministic tie-breaker, not file-arrival order. Review pipelines/customer_cdc.py for AUTO CDC with SCD Type 2. Official Databricks documentation: [R69: AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc).


### Optional feed preparation for the separate AUTO CDC pipeline
This creates a unique append-only synthetic feed. Copy the printed table name into kit.cdc_source when configuring the NEW pipeline. After the first pipeline update succeeds, append sequence 3, run another update, then append sequence 2 and update again. The separate cells are intentional: they let you observe arrival order. [Official AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc).


In [ ]:
PREPARE_CDC_FEED=False
if PREPARE_CDC_FEED:
    import uuid
    CDC_FEED=f"{BRONZE}.customer_feed_{uuid.uuid4().hex[:10]}"
    spark.sql(f"CREATE TABLE {CDC_FEED} (id INT,country STRING,seq BIGINT,op STRING) USING DELTA")
    spark.sql(f"INSERT INTO {CDC_FEED} VALUES (1,'IT',1,'INSERT')")
    print("kit.cdc_source =",CDC_FEED)
    print("After pipeline update, run:", f"INSERT INTO {CDC_FEED} VALUES (1,'FR',3,'UPDATE')")
    print("After the next update, run:", f"INSERT INTO {CDC_FEED} VALUES (1,'DE',2,'UPDATE')")
    print("Inspect the target before testing a delete; source arrival order differs from seq order.")


## 3. AUTO CDC extension
After creating an append-only CDC feed table with id, country, seq and op, add the advanced source to a NEW pipeline and set kit.cdc_source to its fully qualified name. Run events INSERT(id=1,country=IT,seq=1), UPDATE(id=1,country=FR,seq=3), then an older UPDATE(seq=2) to study ordering. Query __START_AT and __END_AT and verify that current attributes follow sequence. Use a delete event and confirm current-row removal while history follows the configured SCD behavior. Official Databricks documentation: [R69: AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc).


## 4. Professional-level discussion prompts
How do you retain enough source history to replay after a week-long outage? How do you make a foreachBatch MERGE idempotent on retries? How do you choose watermark delay using observed lateness? What test catches a late older CDC event overwriting a newer value? How do you compare source and target at the same cutoff? Write a design before coding. These topics are a bridge, not a claim about a future certification blueprint. Official Databricks documentation: [R30: Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints); [R31: Watermarks](https://docs.databricks.com/aws/en/structured-streaming/watermarks); [R68: Change data feed](https://docs.databricks.com/aws/en/tables/features/change-data-feed); [R69: AUTO CDC](https://docs.databricks.com/aws/en/ldp/cdc).
